In [27]:
import pandas as pd
import numpy as np
import joblib

# load dataset 
df = pd.read_excel("fdt_dataset_100_rows.xlsx")
df.head()

,Formulation,Superdisintegrant,Concentration (%),Binder (%),Compression (kN),Disintegration Time (sec)
0,F1,CP,5,2,6,25.0
1,F2,CCS,2,4,5,45.0
2,F3,CP,4,4,7,33.0
3,F4,CP,3,2,5,38.0
4,F5,SSG,3,3,7,35.5


In [28]:
# Prints count of null (missing) values for each column
print("Null values check\n", df.isnull().sum())

Null values check
 Formulation                  0
Superdisintegrant            0
Concentration (%)            0
Binder (%)                   0
Compression (kN)             0
Disintegration Time (sec)    0
dtype: int64


In [39]:
# Prints total number of duplicate rows in the DataFrame
print("Duplicate values check:", df.duplicated().sum())

Duplicate values check: 0


In [30]:
# summary of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Formulation                100 non-null    object 
 1   Superdisintegrant          100 non-null    object 
 2   Concentration (%)          100 non-null    int64  
 3   Binder (%)                 100 non-null    int64  
 4   Compression (kN)           100 non-null    int64  
 5   Disintegration Time (sec)  100 non-null    float64
dtypes: float64(1), int64(3), object(2)
memory usage: 4.8+ KB


In [31]:
# Generates statistical summary (count, mean, std, min, max, quartiles) for numeric columns
df.describe()

,Concentration (%),Binder (%),Compression (kN),Disintegration Time (sec)
count,100.000000,100.000000,100.000000,100.000000
mean,3.630000,3.480000,5.670000,35.150000
std,1.116045,1.123487,1.163893,7.397618
min,2.000000,2.000000,4.000000,21.000000
25%,3.000000,2.000000,5.000000,29.500000
50%,4.000000,3.500000,6.000000,34.500000
75%,5.000000,4.000000,7.000000,40.500000
max,5.000000,5.000000,7.000000,49.500000


In [32]:
# Removes leading and trailing spaces from all column names in the DataFrame
df.columns = df.columns.str.strip()

In [33]:
# Define input and output
X = df.drop(columns=["Formulation", "Disintegration Time (sec)"])
y = df["Disintegration Time (sec)"]

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Columns
categorical_cols = ["Superdisintegrant"]
numeric_cols = ["Concentration (%)", "Binder (%)", "Compression (kN)"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [34]:
from sklearn.linear_model import LinearRegression

lr_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("\n🔹 Linear Regression")
print("MAE:", round(mean_absolute_error(y_test, y_pred_lr), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lr)), 2))
print("R2 Score:", round(r2_score(y_test, y_pred_lr), 2))


🔹 Linear Regression
MAE: 1.64
RMSE: 1.82
R2 Score: 0.93


In [35]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("\n🔹 Random Forest")
print("MAE:", round(mean_absolute_error(y_test, y_pred_rf), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_rf)), 2))
print("R2 Score:", round(r2_score(y_test, y_pred_rf), 2))


🔹 Random Forest
MAE: 1.84
RMSE: 2.35
R2 Score: 0.88


In [36]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", GradientBoostingRegressor(n_estimators=100, random_state=42))
])

gb_pipeline.fit(X_train, y_train)
y_pred_gb = gb_pipeline.predict(X_test)

print("\n🔹 Gradient Boosting")
print("MAE:", round(mean_absolute_error(y_test, y_pred_gb), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_gb)), 2))
print("R2 Score:", round(r2_score(y_test, y_pred_gb), 2))


🔹 Gradient Boosting
MAE: 1.83
RMSE: 2.17
R2 Score: 0.9


In [37]:
# Store R2 scores
model_scores = {
    "Linear Regression": r2_score(y_test, y_pred_lr),
    "Random Forest": r2_score(y_test, y_pred_rf),
    "Gradient Boosting": r2_score(y_test, y_pred_gb)
}

# Find best model
best_model_name = max(model_scores, key=model_scores.get)

print("\n🏆 Best Model:", best_model_name)
print("Best R2 Score:", round(model_scores[best_model_name], 2))


🏆 Best Model: Linear Regression
Best R2 Score: 0.93


In [38]:
if best_model_name == "Linear Regression":
    best_model = lr_pipeline
elif best_model_name == "Random Forest":
    best_model = rf_pipeline
else:
    best_model = gb_pipeline

joblib.dump(best_model, "best_model.pkl")

print("✅ Best model saved as best_model.pkl")

✅ Best model saved as best_model.pkl
